In [ ]:
import networkx as nx
import json


In [ ]:
# Load jsonl file into a list of dictionaries
with open("../data/labelled/all_parallel.jsonl") as test_f:
    data = [json.loads(line) for line in test_f]

In [ ]:
from util import extract_ner_labels

# Create entity and doc id lists, where docs are identified by the doc id
entities, doc_ids = [], []
for item in data:
    tmp = [(e[0].title()) for e in extract_ner_labels(item["text"])]
    entities.append(tmp)
    doc_ids.append(item["id"])

In [ ]:
import matplotlib.pyplot as plt

b = nx.Graph()

# Create ent-doc pairs
edges = []
for pair in zip(entities, doc_ids):
    for ent_list in entities:
        for e in ent_list:
            edges.append((e, pair[1]))

b.add_edges_from(edges)

# Define colors for each entity type
entity_colors = {
    "PER": "#FF6B6B",  # Red
    "LOC": "#4ECDC4",  # Teal
    "ORG": "#95E1D3",  # Light green
    "MISC": "#FFE66D",  # Yellow
}

# Separate nodes by type
doc_nodes = [n for n in b.nodes() if isinstance(n, str) and not isinstance(n, tuple)]
entity_nodes = [n for n in b.nodes() if isinstance(n, tuple)]

# Assign colors to nodes
node_colors = []
for node in b.nodes():
    if isinstance(node, tuple):
        # Entity node - color by type
        entity_type = node[1].upper()
        node_colors.append(entity_colors.get(entity_type, "#CCCCCC"))
    else:
        # Document node - gray
        node_colors.append("#AAAAAA")

# Create larger figure
plt.figure(figsize=(12, 7))

# Use bipartite layout - entities on left, documents on right
pos = nx.bipartite_layout(b, entity_nodes, align="vertical")

# Draw nodes with black outline
nx.draw_networkx_nodes(
    b,
    pos,
    node_color=node_colors,
    node_size=200,
    edgecolors="black",
    linewidths=1,
)

# Draw edges
nx.draw_networkx_edges(b, pos, alpha=0.1, width=0.5)

# Draw labels with offset (below nodes)
labels = {}
for node in b.nodes():
    if isinstance(node, tuple):
        labels[node] = node[0]  # Entity name
    else:
        labels[node] = node  # Document ID

# Draw labels below nodes
label_pos = {k: (v[0], v[1] - 0.01) for k, v in pos.items()}
nx.draw_networkx_labels(b, label_pos, labels, font_size=5)

# Add legend
legend_elements = [
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor=color,
        markersize=10,
        markeredgecolor="black",
        markeredgewidth=2,
        label=entity_type,
    )
    for entity_type, color in entity_colors.items()
]
legend_elements.append(
    plt.Line2D(
        [0],
        [0],
        marker="o",
        color="w",
        markerfacecolor="#AAAAAA",
        markersize=10,
        markeredgecolor="black",
        markeredgewidth=2,
        label="Document",
    )
)
plt.legend(handles=legend_elements, loc="upper right", fontsize=10)

plt.title("Entity-Document Bipartite Graph", fontsize=12)
plt.axis("off")
plt.tight_layout()
plt.show()

print(f"Is bipartite: {nx.is_bipartite(b)}")

In [ ]:
from networkx.algorithms.minors import contracted_nodes
from Levenshtein import distance
import numpy as np

# contracted_nodes(b, "Silvio Greco", "Ruggero Dalla Valle")

nodes = list(b.nodes())
sim_matrix = np.full((len(nodes), len(nodes)), np.inf)

for i in range(len(nodes)):
    for j in range(i, len(nodes)):
        if (isinstance(nodes[i], int)) or (isinstance(nodes[j], int)):
            continue
        else:
            sim_matrix[i, j] = distance(nodes[i], nodes[j])

m = np.matrix(sim_matrix)

In [ ]:
similar_ent = np.argwhere((m <= 2) & (m > 0))

for i, j in similar_ent:
    print(nodes[i], "|", nodes[j], "|", m[i, j])

In [ ]:
b.nodes()